In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/config.json
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/training_args.bin
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/tokenizer.json
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/tokenizer_config.json
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/model.safetensors
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/special_tokens_map.json
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1/vocab.txt
/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202524/1/best_scratch_state.pt
/kaggle/input/emotion-models-20251129-20252

# INFERENCE NOTEBOOK

## Import libraries

In [2]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import shutil

print("Libraries imported.")

Libraries imported.


## Set folders and labels

In [3]:
SCRATCH_FOLDER = "/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202524/1"
BERT_FOLDER    = "/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202529/1"
ROBERTA_FOLDER = "/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202542/1"

# Test CSV folder
DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")

EMOTION_LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

print("Folders set.")

Folders set.


## SCRATCH MODEL INFERENCE

In [4]:
DATA_DIR = "/kaggle/input/2025-sep-dl-gen-ai-project"

# Load data first
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

EMOTION_LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

print("Loaded:", len(train_df), "train rows,", len(test_df), "test rows")


Loaded: 6827 train rows, 1707 test rows


## Load Scratch Model

In [5]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Rebuild the vocab exactly as in training
def build_vocab(texts, max_words=10000):
    c = Counter()
    for t in texts:
        c.update(t.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for i, (w, _) in enumerate(c.most_common(max_words)):
        vocab[w] = i + 2
    return vocab

vocab = build_vocab(train_df["text"].tolist(), max_words=10000)
pad_idx = vocab["<pad>"]
vocab_size = len(vocab)
print("Rebuilt vocab size (scratch):", vocab_size)

# 2) Same encode_text as in training
MAXLEN_SCRATCH = 50

def encode_text(text, vocab, max_len=50):
    words = text.lower().split()
    ids = [vocab.get(w, vocab["<unk>"]) for w in words]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

# 3) Define the SAME SimpleBiLSTM architecture
class SimpleBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_labels=5, pad_idx=0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.fc(pooled)

# 4) Load state dict from Kaggle input folder
SCRATCH_FOLDER = "/kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202524/1"
scratch_model_path = f"{SCRATCH_FOLDER}/best_scratch_state.pt"

print("Loading scratch model from:", scratch_model_path)

model_scratch = SimpleBiLSTM(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128,
    num_labels=len(EMOTION_LABELS),
    pad_idx=pad_idx,
).to(DEVICE)

state = torch.load(scratch_model_path, map_location=DEVICE)
model_scratch.load_state_dict(state)
model_scratch.eval()
print("Scratch model loaded.")

# 5) Build input IDs for all test texts
scratch_inputs = torch.tensor(
    [encode_text(t, vocab, MAXLEN_SCRATCH) for t in test_df["text"]],
    dtype=torch.long
).to(DEVICE)


Rebuilt vocab size (scratch): 10002
Loading scratch model from: /kaggle/input/emotion-models-20251129-202524/pytorch/emotion-models-20251129-202524-20251129-202524/1/best_scratch_state.pt
Scratch model loaded.


## Generate scratch predictions

In [6]:
with torch.no_grad():
    logits = model_scratch(scratch_inputs)              # [num_test, num_labels]
    probs = torch.sigmoid(logits).cpu().numpy()         # move to CPU before numpy
    scratch_preds = (probs > 0.5).astype(int)

print("Scratch predictions done. Shape:", scratch_preds.shape)


Scratch predictions done. Shape: (1707, 5)


## BERT INFERENCE

In [7]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification

TOKEN_MAX_LEN = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_FOLDER)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_FOLDER)
bert_model.to(DEVICE)
bert_model.eval()

print("BERT model loaded.")

2025-11-30 02:39:45.029044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764470385.240935      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764470385.300716      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


BERT model loaded.


In [8]:
# Tokenize test text for BERT
bert_enc = bert_tokenizer(
    list(test_df["text"]),
    padding="max_length",
    truncation=True,
    max_length=TOKEN_MAX_LEN,
    return_tensors="pt"
)

# Send to device
bert_enc = {k: v.to(DEVICE) for k, v in bert_enc.items()}

# Predictions
with torch.no_grad():
    logits = bert_model(**bert_enc).logits
    probs = torch.sigmoid(logits).cpu().numpy()
    bert_preds = (probs > 0.5).astype(int)

print("BERT predictions done.")


BERT predictions done.


## ROBERTA INFERENCE

In [9]:

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_FOLDER)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_FOLDER)
roberta_model.to(DEVICE)
roberta_model.eval()

print("RoBERTa model loaded.")

RoBERTa model loaded.


In [10]:
# Tokenize test text for RoBERTa
roberta_enc = roberta_tokenizer(
    list(test_df["text"]),
    padding="max_length",
    truncation=True,
    max_length=TOKEN_MAX_LEN,
    return_tensors="pt"
)

roberta_enc = {k: v.to(DEVICE) for k, v in roberta_enc.items()}

with torch.no_grad():
    logits = roberta_model(**roberta_enc).logits
    probs = torch.sigmoid(logits).cpu().numpy()
    roberta_preds = (probs > 0.5).astype(int)

print("RoBERTa predictions done.")


RoBERTa predictions done.


## SAVE ALL SUBMISSIONS

In [11]:
# Scratch submission
sub_s = pd.DataFrame(scratch_preds, columns=EMOTION_LABELS)
sub_s.insert(0, "id", test_df["id"])
sub_s.to_csv("submission_scratch.csv", index=False)

# BERT submission
sub_b = pd.DataFrame(bert_preds, columns=EMOTION_LABELS)
sub_b.insert(0, "id", test_df["id"])
sub_b.to_csv("submission_bert.csv", index=False)

# RoBERTa submission
sub_r = pd.DataFrame(roberta_preds, columns=EMOTION_LABELS)
sub_r.insert(0, "id", test_df["id"])
sub_r.to_csv("submission_roberta.csv", index=False)

print("All submissions saved.")

All submissions saved.
